In [1]:
!pip install -q transformers[torch] datasets evaluate accelerate


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import random
import evaluate
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

In [ ]:
# Update filename if it differs from the notebook
path = "data/final_clean_dataset.csv"
df = pd.read_csv(path)

# Remove duplicates and empty entries
df = df.drop_duplicates(subset=["text"])
df = df[df["text"].notna() & (df["text"].str.strip() != "")]

# Encode labels
df["label"], label_names = pd.factorize(df["class_name"])
num_labels = len(label_names)

print(f"Total unique samples: {len(df)}")
print(f"Number of classes: {num_labels}")

def remove_label_leakage(row):
    text = str(row["text"]).lower()
    class_name = str(row["class_name"]).replace("_", " ").lower()
    text = text.replace(class_name, "")
    return " ".join(text.split())

df["text"] = df.apply(remove_label_leakage, axis=1)

# Drop very short rows after leakage removal
df = df[df["text"].str.len() > 10]


Total unique samples: 2825
Number of classes: 40


In [5]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
val_dataset = Dataset.from_pandas(val_df[["text", "label"]])

In [6]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2260 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

# Apply custom dropout
model.config.dropout = 0.1
model.config.attention_dropout = 0.1


pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect i

In [9]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/plant_training/results_tanya",

    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,

    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=42,
)


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Score
1,3.696104,3.665317,0.024779,0.001777
2,3.639613,3.571814,0.107965,0.052288
3,3.545705,3.438993,0.246018,0.187777
4,3.432069,3.310001,0.376991,0.336502
5,3.324375,3.195746,0.479646,0.453628
6,3.223371,3.097871,0.523894,0.504681
7,3.138591,3.013497,0.550442,0.533541
8,3.058393,2.931310,0.575221,0.562558
9,2.987633,2.859456,0.592920,0.581877
10,2.923122,2.798443,0.605310,0.589721


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.encoder.layer.0.output.LayerNorm.beta', 'bert.encoder.layer.0.output.LayerNorm.gamma', 'bert.encoder.layer.1.attention.output.LayerNorm.beta', 'bert.encoder.layer.1.attention.output.LayerNorm.gamma', 'bert.en

TrainOutput(global_step=5660, training_loss=3.005817405242381, metrics={'train_runtime': 107.8997, 'train_samples_per_second': 418.908, 'train_steps_per_second': 52.456, 'total_flos': 14526669619200.0, 'train_loss': 3.005817405242381, 'epoch': 20.0})

In [11]:
# Evaluate the best model
eval_results = trainer.evaluate()
print(f"Final Evaluation: {eval_results}")

# Save the model and tokenizer to Drive
model_save_path = "/content/drive/MyDrive/plant_training/bert_model_final"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"✅ Model saved to {model_save_path}")

Final Evaluation: {'eval_loss': 2.5168933868408203, 'eval_accuracy': 0.6778761061946903, 'eval_f1_score': 0.6885952951536414, 'eval_runtime': 0.3379, 'eval_samples_per_second': 1671.956, 'eval_steps_per_second': 210.104, 'epoch': 20.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to /content/drive/MyDrive/plant_training/bert_model_final


In [12]:
import torch

# 1. Load the saved model and tokenizer
model_path = "/content/drive/MyDrive/plant_training/bert_model_final"
inference_model = AutoModelForSequenceClassification.from_pretrained(model_path)
inference_tokenizer = AutoTokenizer.from_pretrained(model_path)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inference_model.to(device)
inference_model.eval()

def predict_plant_disease(text):
    # 2. Tokenize the input text
    inputs = inference_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    ).to(device)

    # 3. Get prediction
    with torch.no_grad():
        outputs = inference_model(**inputs)
        logits = outputs.logits
        probabilities = torch.nn.functional.softmax(logits, dim=-1)
        predicted_class_id = torch.argmax(probabilities, dim=-1).item()
        confidence = probabilities[0][predicted_class_id].item()

    # 4. Map ID back to name (using the label_names from your earlier cell)
    disease_name = label_names[predicted_class_id]

    return disease_name, confidence

# --- Test it out ---
test_text = "The leaves of my tomato plant have small brown spots with yellow halos."
disease, score = predict_plant_disease(test_text)

print(f"Predicted Disease: {disease}")
print(f"Confidence: {score:.2%}")

Loading weights:   0%|          | 0/41 [00:00<?, ?it/s]

Predicted Disease: Tomato_Yellow_Leaf_Curl_Virus
Confidence: 6.75%


In [13]:
# Check training logs
for log in trainer.state.log_history:
    if 'loss' in log:
        print(f"Step: {log['step']}, Loss: {log['loss']:.4f}")
    if 'eval_accuracy' in log:
        print(f"--- Eval Acc: {log['eval_accuracy']:.4f} ---")

Step: 283, Loss: 3.6961
--- Eval Acc: 0.0248 ---
Step: 566, Loss: 3.6396
--- Eval Acc: 0.1080 ---
Step: 849, Loss: 3.5457
--- Eval Acc: 0.2460 ---
Step: 1132, Loss: 3.4321
--- Eval Acc: 0.3770 ---
Step: 1415, Loss: 3.3244
--- Eval Acc: 0.4796 ---
Step: 1698, Loss: 3.2234
--- Eval Acc: 0.5239 ---
Step: 1981, Loss: 3.1386
--- Eval Acc: 0.5504 ---
Step: 2264, Loss: 3.0584
--- Eval Acc: 0.5752 ---
Step: 2547, Loss: 2.9876
--- Eval Acc: 0.5929 ---
Step: 2830, Loss: 2.9231
--- Eval Acc: 0.6053 ---
Step: 3113, Loss: 2.8682
--- Eval Acc: 0.6142 ---
Step: 3396, Loss: 2.8194
--- Eval Acc: 0.6283 ---
Step: 3679, Loss: 2.7773
--- Eval Acc: 0.6389 ---
Step: 3962, Loss: 2.7399
--- Eval Acc: 0.6602 ---
Step: 4245, Loss: 2.7053
--- Eval Acc: 0.6584 ---
Step: 4528, Loss: 2.6808
--- Eval Acc: 0.6637 ---
Step: 4811, Loss: 2.6573
--- Eval Acc: 0.6708 ---
Step: 5094, Loss: 2.6449
--- Eval Acc: 0.6761 ---
Step: 5377, Loss: 2.6276
--- Eval Acc: 0.6779 ---
Step: 5660, Loss: 2.6267
--- Eval Acc: 0.6779 ---
---